<a href="https://colab.research.google.com/github/clin0603/csci3052u-group-project/blob/main/milestone3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Milestone 3 - Baseline

In [ ]:
# imports
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

## Data Source


In [ ]:
# Create folder in Colab
!mkdir -p Random_Mortgage_Data
# Download randomized datasets into the folder
!wget -q -O Random_Mortgage_Data/Random_Mortgage_Data07.csv https://raw.githubusercontent.com/clin0603/csci3052u-group-project/main/Random_Mortgage_Data/Random_Mortgage_Data07.csv
!wget -q -O Random_Mortgage_Data/Random_Mortgage_Data08.csv https://raw.githubusercontent.com/clin0603/csci3052u-group-project/main/Random_Mortgage_Data/Random_Mortgage_Data08.csv
!wget -q -O Random_Mortgage_Data/Random_Mortgage_Data09.csv https://raw.githubusercontent.com/clin0603/csci3052u-group-project/main/Random_Mortgage_Data/Random_Mortgage_Data09.csv

In [ ]:
file_paths = [
    "Random_Mortgage_Data/Random_Mortgage_Data07.csv",
    "Random_Mortgage_Data/Random_Mortgage_Data08.csv",
    "Random_Mortgage_Data/Random_Mortgage_Data09.csv"
]

df = pd.concat(
    [pd.read_csv(file, low_memory=False) for file in file_paths],
    ignore_index=True
)

print("Dataset shape:", df.shape)

Dataset shape: (120000, 78)


# Milestone 3: Nevada mortgage approval baseline
**Team Remedial | Group 35 | CSCI-3052U-002**

Train: 2007. Validation: 2008. Final test: 2009. The primary data are the frozen randomized CSVs already used in the team repository. The older first-record sample is a sensitivity experiment.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
if not Path('src/baseline.py').exists():
    raise RuntimeError('Open this notebook with the working directory set to the repository root.')
from src.baseline import run, FEATURES
from src.download_data import main as verify_data
print('Python:', sys.version.split()[0])
print('Features:', FEATURES)
verify_data()

## 2. Working pipeline and baselines
Actions 1 and 2 map to approved (1); action 3 maps to denied (0); other outcomes are excluded. Only loan amount, income, loan type and loan purpose enter the model. Numeric medians, missing indicators, scaling and one-hot encoding are fitted on 2007 only.

Majority classifier: always predicts the 2007 majority class. Course baseline: Euclidean kNN, uniform weights. Select k from 5, 15, 31, 61, 101 using 2008 ROC-AUC; use smaller k to break exact ties. Freeze k before testing 2009; do not refit on 2008.

In [ ]:
primary = run('random')
display(pd.read_csv('results/random/data_audit.csv'))
display(pd.read_csv('results/random/validation_search.csv'))
display(primary.round(4))

## 3. Primary metric and error analysis
ROC-AUC uses approval=1 and probability of approval. Denial precision, recall and F1 explicitly treat denied=0 as the class of interest. Accuracy alone can hide a failure to detect denials. The confusion counts preserve both classes.

In [ ]:
display(primary[['split','model','auc','accuracy','balanced_accuracy','denial_precision','denial_recall','denial_f1']].round(4))
display(primary[['split','model','denied_as_denied','denied_as_approved','approved_as_denied','approved_as_approved']])
display(Image(filename='results/random/test_roc.png'))

## 4. Sensitivity to the older Milestone 2 sample
This separate experiment uses the first-record CSVs documented in the earlier memo. It has its own validation-only k selection and is not used to choose the primary model. Different sampling and different year sizes prevent attributing any metric difference solely to the crisis.

In [ ]:
sensitivity = run('cut')
display(pd.read_csv('results/cut/data_audit.csv'))
display(sensitivity[['split','model','auc','accuracy','denial_recall','denial_f1']].round(4))

## 5. Start / stop / continue
**Start:** document the original sampling procedure and add lender-aware uncertainty analysis in the next milestone. Compare logistic regression using the same chronology.

**Stop:** selecting first records, treating category codes as ordered distances, and inferring crisis causation or credit risk from approval classification.

**Continue:** train-only preprocessing, chronological evaluation, checksummed data and explicit minority-class metrics.

Sources: team repository at commit `0e570dbc60bc705096ace8a05cad877aff781864`; [CFPB historical HMDA data](https://www.consumerfinance.gov/data-research/hmda/historic-data/); [HMDA code sheet](https://files.consumerfinance.gov/hmda-historic-data-dictionaries/lar_record_codes.pdf).

In [ ]:
import unittest
suite = unittest.defaultTestLoader.discover('tests')
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()